In [ ]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
import numpy as np

df_raw = pd.read_csv(os.path.join("./","merged_daily.csv"))

df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw['Date_CDP'] = pd.to_datetime(df_raw['Date_CDP'])
df_raw['weekday'] = df_raw['date'].dt.weekday  # 0 (lunes) a 6 (domingo)

features = [
    'mean_added_count', 'median_added_count', 'max_added_count', 'min_added_count',
    'mean_removed_count', 'median_removed_count', 'max_removed_count', 'min_removed_count',
    'mean_index', 'median_index', 'max_index', 'min_index',
    'month',
    'Edad', 'Genero', 'FAME_n_High', 'FAME_n_Low', 'FAME_n_No',
    'Tipo_EM_EMPP', 'Tipo_EM_EMRR', 'Tipo_EM_EMSP'
]
# 'year','month', 'week', 'weekday', 'day',
# TODO: Puedo quitar ruido con dias

sequences = []
labels = []

seq_len = 30  # Longitud de la secuencia
pred_window = 30
for key, group in df_raw.groupby('key'):
    group = group.sort_values('date').reset_index(drop=True)
    pira_date = group['Date_CDP'].dropna().unique()
    pira_date = pd.to_datetime(pira_date[0]) if len(pira_date) > 0 else None

    for i in range(len(group) - seq_len - pred_window + 1):
        window = group.iloc[i:i + seq_len]
        future = group.iloc[i + seq_len:i + seq_len + pred_window]

        X_window = window[features].values.astype(np.float32)

        if pira_date and any((future['date'] >= pira_date)):
            label = 1
        else:
            label = 0

        sequences.append(X_window)
        labels.append(label)

        
X_arr = np.array(sequences)
y_arr = np.array(labels)

# Contar cuantos 0 y 1 hay en mi numpy array y_arr
unique, counts = np.unique(y_arr, return_counts=True)
print(f"Unique labels in y_arr: {dict(zip(unique, counts))}")

N, T, D = X_arr.shape
reshaped = X_arr.reshape(-1, D)

# Reemplazar inf por NaN
reshaped[np.isinf(reshaped)] = np.nan

# Detectar NaNs y reemplazar con la media de la columna
if np.isnan(reshaped).any():
    print(f"⚠️ Warning: NaNs detected in data. Replacing with column means...")
    col_means = np.nanmean(reshaped, axis=0)
    inds = np.where(np.isnan(reshaped))
    reshaped[inds] = np.take(col_means, inds[1])

# Escalar
scaler = StandardScaler()

from sklearn.model_selection import train_test_split

# Primer split: train vs temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_arr, y_arr, test_size=0.3, random_state=42, stratify=y_arr)

# Segundo split: val vs test (50% y 50% de temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

X_train_scaled = scaler.fit_transform(X_train.reshape(-1, D))
N_train = X_train.shape[0]  # Número de muestras en el conjunto de entrenamiento
X_train = X_train_scaled.reshape(N_train, T, D)

X_val_scaled = scaler.transform(X_val.reshape(-1, D))
N_val = X_val.shape[0]  # Número de muestras en el conjunto de validación
X_val = X_val_scaled.reshape(N_val, T, D)

X_test_scaled = scaler.transform(X_test.reshape(-1, D))
N_test = X_test.shape[0]  # Número de muestras en el conjunto de test
X_test = X_test_scaled.reshape(N_test, T, D)

X_train = X_train
y_train = y_train
X_val = X_val
y_val = y_val
X_test = X_test
y_test = y_test
print('train shape:', X_train.shape, y_train.shape)
print('val shape:', X_val.shape, y_val.shape)
print('test shape:', X_test.shape, y_test.shape)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class TimeSeriesClassificationDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx])
        y = torch.tensor(self.y[idx])  # ← sin corchetes
        return x, y
    
train_dataset = TimeSeriesClassificationDataset(X_train, y_train)
val_dataset = TimeSeriesClassificationDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
import torch
import torch.nn as nn

class TemporalTransformerClassifier(nn.Module):
    def __init__(self, input_dim=21, seq_len=30, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.d_model = d_model

        self.input_proj = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))  # [CLS]
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_len + 1, d_model))  # +1 for [CLS]

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):  # x: (batch_size, seq_len, input_dim)
        batch_size = x.size(0)

        # Project input features to model dimension
        x = self.input_proj(x)  # (batch_size, seq_len, d_model)

        # Expand CLS token and prepend to input sequence
        cls_token = self.cls_token.expand(batch_size, 1, -1)  # (batch_size, 1, d_model)
        x = torch.cat([cls_token, x], dim=1)  # (batch_size, seq_len + 1, d_model)

        # Add positional embeddings
        x = x + self.pos_embedding[:, :x.size(1), :]  # (batch_size, seq_len + 1, d_model)

        # Pass through transformer
        x = self.transformer(x)  # (batch_size, seq_len + 1, d_model)

        # Use CLS token output
        cls_output = x[:, 0]  # (batch_size, d_model)

        return self.classifier(cls_output)  # (batch_size, 1)

In [ ]:
import torch.nn.functional as F
from torch.optim import Adam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TemporalTransformerClassifier().to(device)
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr=1e-4)

def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).squeeze()
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(dataloader.dataset)

def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).squeeze()
            loss = criterion(pred, yb)
            total_loss += loss.item() * xb.size(0)
            predicted = (pred > 0.5).float()
            correct += (predicted == yb).sum().item()
    acc = correct / len(dataloader.dataset)
    return total_loss / len(dataloader.dataset), acc


In [ ]:
EPOCHS = 20

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}: Train loss={train_loss:.4f}, Val loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

In [ ]:
test_dataset = TimeSeriesClassificationDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32)

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Asegúrate de que el modelo esté en modo evaluación
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        outputs = model(xb).squeeze()  # (batch,)
        preds = (outputs > 0.5).float()  # umbral binario
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(yb.cpu().numpy())

cm = confusion_matrix(all_targets, all_preds)
print("Matriz de confusión:")
print(cm)

# Opcional: imprimir métricas detalladas
print("\nReporte de clasificación:")
print(classification_report(all_targets, all_preds, digits=4))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No PIRA", "PIRA"], yticklabels=["No PIRA", "PIRA"])
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión")
plt.show()